# Notebook 04 — Combined Live Risk Query

This is the single entry point matching the actual product goal: **type one place name,
get flood risk, landslide risk, and (if available) the most recent damage assessment for
that area, together.** This is exactly what the backend API (see the Cursor prompt) should
replicate as an endpoint -- this notebook is effectively that endpoint's logic, runnable
standalone for testing before it's wired into the actual website.

Loads the already-trained flood and landslide models (Notebooks 01 and 02 must be run
first) plus their saved honest-accuracy figures, and combines both into one query.

In [1]:
!pip install -q requests pandas scikit-learn joblib geopandas

import requests, json
import pandas as pd, numpy as np
import joblib
import geopandas as gpd
from datetime import date, timedelta
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/DIP_Kerala'

Mounted at /content/drive


## Load boundaries, feature store, both trained models, and their honest accuracy figures

In [2]:
lsgd = gpd.read_file(f'{BASE}/00_boundaries/kerala_lsgd_boundaries.geojson')
rename_map = {'local_auth': 'lsgd_type', 'name': 'lsgd_name', 'District': 'district'}
lsgd = lsgd.rename(columns={k: v for k, v in rename_map.items() if k in lsgd.columns})
if 'lsgd_id' not in lsgd.columns:
    lsgd['lsgd_id'] = range(1, len(lsgd) + 1)

lsgd_wgs84 = lsgd.to_crs(epsg=4326)
lsgd_wgs84['centroid'] = lsgd_wgs84.geometry.centroid
lsgd_wgs84['lat'] = lsgd_wgs84['centroid'].y
lsgd_wgs84['lon'] = lsgd_wgs84['centroid'].x

feature_store = pd.read_csv(f'{BASE}/03_processed/lsgd_feature_store.csv')

flood_model = joblib.load(f'{BASE}/05_models/flood_model.pkl')
flood_scaler = joblib.load(f'{BASE}/05_models/flood_scaler.pkl')
landslide_model = joblib.load(f'{BASE}/05_models/landslide_model.pkl')
landslide_scaler = joblib.load(f'{BASE}/05_models/landslide_scaler.pkl')
landslide_le = joblib.load(f'{BASE}/05_models/landslide_label_encoder.pkl')

def load_accuracy(fname):
    try:
        with open(f'{BASE}/05_models/{fname}') as f:
            return json.load(f)
    except FileNotFoundError:
        return None

flood_accuracy = load_accuracy('flood_accuracy.json')
landslide_accuracy = load_accuracy('landslide_accuracy.json')
damage_accuracy = load_accuracy('damage_assessment_accuracy.json')

print(f"Loaded {len(lsgd_wgs84)} LSGD units")
print(f"Flood model honest accuracy: {flood_accuracy['honest_grouped_accuracy_mean']:.4f}" if flood_accuracy else "Flood accuracy file not found -- run Notebook 01 first")
print(f"Landslide model honest accuracy: {landslide_accuracy['honest_grouped_accuracy_mean']:.4f}" if landslide_accuracy else "Landslide accuracy file not found -- run Notebook 02 first")
print(f"Damage assessment accuracy: {damage_accuracy['overall_accuracy']:.4f}" if damage_accuracy else "Damage assessment accuracy file not found -- run Notebook 03 first")

/tmp/ipykernel_5334/1025063954.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  lsgd_wgs84['centroid'] = lsgd_wgs84.geometry.centroid


Loaded 1034 LSGD units
Flood model honest accuracy: 0.5084
Landslide model honest accuracy: 0.1773
Damage assessment accuracy: 0.6600


## Live weather fetch (shared by both models)

In [3]:
def fetch_live_weather(lat, lon):
    end_date = date.today()
    start_date = end_date - timedelta(days=6)
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date.isoformat(), "end_date": end_date.isoformat(),
        "daily": "precipitation_sum",
        "current": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "timezone": "auto",
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    data = r.json()
    daily = data.get('daily', {})
    values = [v for v in daily.get('precipitation_sum', []) if v is not None]
    return {
        'rainfall_7day_mm': sum(values) if values else 0.0,
        'daily_breakdown': dict(zip(daily.get('time', []), daily.get('precipitation_sum', []))),
        'current': data.get('current', {}),
    }

## The combined query — this is the function the backend API endpoint should mirror

In [4]:
flood_feature_cols = ['elevation', 'slope', 'rainfall_7day_mm', 'dist_to_water_m', 'vegetation', 'builtup']
landslide_feature_cols = ['elevation', 'slope', 'rainfall_7day_mm', 'vegetation', 'dist_to_water_m']

def get_area_risk(place_name):
    row = lsgd_wgs84[lsgd_wgs84['lsgd_name'].str.lower() == place_name.lower()]
    if row.empty:
        row = lsgd_wgs84[lsgd_wgs84['lsgd_name'].str.lower().str.contains(place_name.lower(), na=False)]
    if row.empty:
        print(f"'{place_name}' not found in Kerala's 1,034 LSGD units.")
        print("This is exactly the case the website's 'add a missing place' feature should catch --")
        print("see the Cursor prompt's database design for how a user-submitted place should be handled.")
        return None
    row = row.iloc[0]

    static = feature_store[feature_store['lsgd_id'] == row['lsgd_id']]
    if static.empty:
        print(f"No feature store entry for lsgd_id {row['lsgd_id']}.")
        return None
    static = static.iloc[0]

    weather = fetch_live_weather(row['lat'], row['lon'])
    current = weather['current']

    flood_input = pd.DataFrame([{
        'elevation': static['elevation'], 'slope': static['slope'],
        'rainfall_7day_mm': weather['rainfall_7day_mm'],
        'dist_to_water_m': static['dist_to_water_m'],
        'vegetation': static['vegetation'], 'builtup': static['builtup'],
    }])[flood_feature_cols]
    landslide_input = flood_input[landslide_feature_cols]

    flood_prob = flood_model.predict_proba(flood_scaler.transform(flood_input))[0][1]
    landslide_pred = landslide_model.predict(landslide_scaler.transform(landslide_input))[0]
    landslide_risk = landslide_le.inverse_transform([landslide_pred])[0]
    landslide_conf = landslide_model.predict_proba(landslide_scaler.transform(landslide_input))[0].max()

    def flood_risk_label(p):
        return "HIGH" if p > 0.7 else "MODERATE" if p > 0.4 else "LOW"

    result = {
        'place': row['lsgd_name'],
        'type': row['lsgd_type'],
        'district': row['district'],
        'lat': row['lat'], 'lon': row['lon'],
        'weather': {
            'temperature_c': current.get('temperature_2m'),
            'humidity_pct': current.get('relative_humidity_2m'),
            'rainfall_7day_mm': weather['rainfall_7day_mm'],
            'daily_breakdown': weather['daily_breakdown'],
        },
        'flood': {'probability': float(flood_prob), 'risk_level': flood_risk_label(flood_prob)},
        'landslide': {'risk_level': landslide_risk, 'confidence': float(landslide_conf)},
        'damage_assessment': 'Not available for this query -- requires a manually supplied pre/post satellite image pair (see Notebook 03). This is a separate, decoupled workflow, not triggered by a place-name search.',
    }

    print(f"Location: {result['place']} ({result['type']}, {result['district']} district)")
    print(f"\n--- Live Weather ---")
    print(f"Temperature: {result['weather']['temperature_c']} C | Humidity: {result['weather']['humidity_pct']}%")
    print(f"Rainfall, past 7 days: {result['weather']['rainfall_7day_mm']:.1f} mm")
    print(f"\n--- Flood Risk ---")
    print(f"Probability: {result['flood']['probability']*100:.1f}%  ->  {result['flood']['risk_level']}")
    print(f"\n--- Landslide Risk ---")
    print(f"Risk Level: {result['landslide']['risk_level']}  (confidence: {result['landslide']['confidence']*100:.1f}%)")
    print(f"\n--- Damage Assessment ---")
    print(result['damage_assessment'])

    return result

# Try it:
# result = get_area_risk("Chengannur")

## Model Accuracy Reference (loaded from the saved JSON files, not recomputed here)

In [5]:
print("="*50)
print("MODEL ACCURACY REFERENCE")
print("="*50)
if flood_accuracy:
    print(f"Flood model ({flood_accuracy['model']}): {flood_accuracy['honest_grouped_accuracy_mean']:.4f} (honest, grouped)")
if landslide_accuracy:
    print(f"Landslide model ({landslide_accuracy['model']}): {landslide_accuracy['honest_grouped_accuracy_mean']:.4f} (honest, grouped)")
if damage_accuracy:
    print(f"Damage assessment: {damage_accuracy['overall_accuracy']:.4f} overall, {damage_accuracy['macro_f1']:.4f} macro-F1")
print("="*50)
print("\nNote: this notebook doesn't train anything, so there's no new accuracy to compute --")
print("these figures are read from what Notebooks 01/02/03 already saved after training.")

MODEL ACCURACY REFERENCE
Flood model (RandomForest): 0.5084 (honest, grouped)
Landslide model (XGBoost): 0.1773 (honest, grouped)
Damage assessment: 0.6600 overall, 0.5852 macro-F1

Note: this notebook doesn't train anything, so there's no new accuracy to compute --
these figures are read from what Notebooks 01/02/03 already saved after training.


## Batch mode — scan all 1,034 units (what a scheduled backend job would run for alerts)

In [6]:
def scan_all_units(flood_threshold=0.7, limit=None):
    results = []
    units = lsgd_wgs84 if limit is None else lsgd_wgs84.head(limit)
    for _, row in units.iterrows():
        static = feature_store[feature_store['lsgd_id'] == row['lsgd_id']]
        if static.empty:
            continue
        static = static.iloc[0]
        weather = fetch_live_weather(row['lat'], row['lon'])
        flood_input = pd.DataFrame([{
            'elevation': static['elevation'], 'slope': static['slope'],
            'rainfall_7day_mm': weather['rainfall_7day_mm'],
            'dist_to_water_m': static['dist_to_water_m'],
            'vegetation': static['vegetation'], 'builtup': static['builtup'],
        }])[flood_feature_cols]
        flood_prob = flood_model.predict_proba(flood_scaler.transform(flood_input))[0][1]
        results.append({'lsgd_name': row['lsgd_name'], 'district': row['district'], 'flood_probability': flood_prob})
    results_df = pd.DataFrame(results)
    alerts = results_df[results_df['flood_probability'] > flood_threshold]
    print(f"Scanned {len(results_df)} units. {len(alerts)} above threshold.")
    return results_df, alerts

# Test on a small subset first (each unit = 1 API call):
# results_df, alerts = scan_all_units(limit=20)